In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd

df = pd.read_csv(
    "Movie_revenue_dataset.csv",
    engine="python",
    on_bad_lines="skip"
)

print(df.shape)

(2705, 12)


In [ ]:
df.head(2)

,Unnamed: 0,vote_average,vote_count,release_date,revenue,runtime,budget,original_language,overview,popularity,genre,spoken_languages
0,0,8.364,34495.0,2010-07-15,825532764.0,148.0,160000000.0,en,"Cobb, a skilled thief who commits corporate es...",83.952,"Action, Science Fiction, Adventure","English, French, Japanese, Swahili"
1,1,8.417,32571.0,2014-11-05,701729206.0,169.0,165000000.0,en,The adventures of a group of explorers who mak...,140.241,"Adventure, Drama, Science Fiction",English


In [ ]:
df.shape

(2705, 12)

In [ ]:
df["movie_success"] = "Flop"

df.loc[
    (df["budget"] > 0) &
    (df["revenue"] >= df["budget"]) &
    (df["revenue"] < 2 * df["budget"]),
    "movie_success"
] = "Average"

df.loc[
    (df["budget"] > 0) &
    (df["revenue"] >= 2 * df["budget"]),
    "movie_success"
] = "Hit"

print(df["movie_success"].value_counts())

movie_success
Hit        1804
Flop        455
Average     446
Name: count, dtype: int64


In [ ]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

df["year"] = df["release_date"].dt.year
df["month"] = df["release_date"].dt.month

df["year"] = df["year"].fillna(df["year"].median())
df["month"] = df["month"].fillna(df["month"].mode()[0])

In [ ]:

df["budget_per_runtime"] = df["budget"] / (df["runtime"] + 1)

df["popularity_per_vote"] = df["popularity"] / (df["vote_count"] + 1)


df["budget_log"] = np.log1p(df["budget"])
df["vote_count_log"] = np.log1p(df["vote_count"])
df["popularity_log"] = np.log1p(df["popularity"])


current_year = int(df["year"].max())
df["movie_age"] = current_year - df["year"]

In [ ]:
df.head(2)

,Unnamed: 0,vote_average,vote_count,release_date,revenue,runtime,budget,original_language,overview,popularity,...,spoken_languages,movie_success,year,month,budget_per_runtime,popularity_per_vote,budget_log,vote_count_log,popularity_log,movie_age
0,0,8.364,34495.0,2010-07-15,825532764.0,148.0,160000000.0,en,"Cobb, a skilled thief who commits corporate es...",83.952,...,"English, French, Japanese, Swahili",Hit,2010,7,1.073826e+06,0.002434,18.890684,10.448599,4.442086,13
1,1,8.417,32571.0,2014-11-05,701729206.0,169.0,165000000.0,en,The adventures of a group of explorers who mak...,140.241,...,English,Hit,2014,11,9.705882e+05,0.004306,18.921456,10.391208,4.950468,9


In [ ]:
df.shape

(2705, 21)

In [ ]:
df['movie_success'].value_counts()

,count
movie_success,
Hit,1804
Flop,455
Average,446


In [ ]:
num_cols = [
    "vote_average",
    "vote_count",
    "revenue",
    "runtime",
    "budget",
    "popularity"
]

for col in num_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}")
    print(f"Outliers : {len(outliers)}")
    print("-"*40)

vote_average
Outliers : 10
----------------------------------------
vote_count
Outliers : 216
----------------------------------------
revenue
Outliers : 207
----------------------------------------
runtime
Outliers : 71
----------------------------------------
budget
Outliers : 164
----------------------------------------
popularity
Outliers : 211
----------------------------------------


In [ ]:
Q1 = df["runtime"].quantile(0.25)
Q3 = df["runtime"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[
    (df["runtime"] >= lower) &
    (df["runtime"] <= upper)
]

In [ ]:
df.describe()

,Unnamed: 0,vote_average,vote_count,release_date,revenue,runtime,budget,popularity,year,month,budget_per_runtime,popularity_per_vote,budget_log,vote_count_log,popularity_log,movie_age
count,2634.000000,2634.000000,2634.000000,2634,2.634000e+03,2634.000000,2.634000e+03,2634.000000,2634.000000,2634.000000,2.634000e+03,2634.000000,2634.000000,2634.000000,2634.000000,2634.000000
mean,1360.925209,6.859561,4761.676158,2006-11-29 17:01:13.804100096,1.845521e+08,111.319666,5.192042e+07,36.335513,2006.369021,7.047077,4.513551e+05,0.010246,16.138212,8.222352,3.389870,16.630979
min,0.000000,2.890000,1563.000000,1921-01-21 00:00:00,0.000000e+00,60.000000,0.000000e+00,0.600000,1921.000000,1.000000,0.000000e+00,0.000119,0.000000,7.355002,0.470004,0.000000
25%,690.250000,6.350000,2143.000000,2001-11-22 18:00:00,3.866737e+07,98.000000,1.180000e+07,19.828750,2001.000000,4.000000,1.058104e+05,0.005296,16.283610,7.670429,3.036334,7.000000
50%,1362.500000,6.855500,3269.000000,2010-03-26 00:00:00,1.050594e+08,109.000000,3.300000e+07,26.835500,2010.000000,7.000000,2.912621e+05,0.007731,17.312018,8.092545,3.326312,13.000000
75%,2033.750000,7.391500,5697.500000,2016-03-29 00:00:00,2.325562e+08,123.000000,7.500000e+07,38.376750,2016.000000,10.000000,6.547018e+05,0.010898,18.132999,8.647958,3.673176,22.000000
max,2704.000000,8.702000,34495.000000,2023-08-02 00:00:00,2.923706e+09,163.000000,3.790000e+08,1567.273000,2023.000000,12.000000,2.746377e+06,0.770159,19.753047,10.448599,7.357730,102.000000
std,777.112533,0.727780,4062.002712,NaN,2.399088e+08,17.761928,5.559532e+07,57.404580,13.727328,3.354154,4.623364e+05,0.021855,4.357486,0.655896,0.578900,13.727328


In [ ]:
df.isnull().sum()

,0
Unnamed: 0,0
vote_average,0
vote_count,0
release_date,0
revenue,0
runtime,0
budget,0
original_language,0
overview,0
popularity,0


In [ ]:
df = df.drop(columns=["overview","genre","spoken_languages","release_date","Unnamed: 0","original_language","revenue"])

KeyError: "['overview', 'genre', 'spoken_languages', 'release_date', 'Unnamed: 0', 'original_language', 'revenue'] not found in axis"

In [ ]:
df["movie_success"] = df["movie_success"].map({
    "Flop": 0,
    "Average": 1,
    "Hit": 2
})

In [ ]:
X = df.drop(["movie_success"], axis=1)
y = df['movie_success']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

ValueError: Input y contains NaN.

In [ ]:
print(df["movie_success"].unique())

[2 1 0]


In [ ]:
from sklearn.utils import resample
import pandas as pd


train_df = pd.concat([X_train, y_train], axis=1)


class_0 = train_df[train_df["movie_success"] == 0]
class_1 = train_df[train_df["movie_success"] == 1]
class_2 = train_df[train_df["movie_success"] == 2]

class_0 = resample(
    class_0,
    replace=False,
    n_samples=300000,
    random_state=42
)


class_1 = resample(
    class_1,
    replace=True,
    n_samples=150000,
    random_state=42
)

class_2 = resample(
    class_2,
    replace=True,
    n_samples=150000,
    random_state=42
)


train_df = pd.concat(
    [class_0, class_1, class_2],
    ignore_index=True
)


train_df = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


X_train = train_df.drop("movie_success", axis=1)
y_train = train_df["movie_success"]

print(y_train.value_counts())

ValueError: Cannot sample 300000 out of arrays with dim 354 when replace is False

In [ ]:
X_train = train_df.drop("movie_success", axis=1)
y_train = train_df["movie_success"]

NameError: name 'train_df' is not defined

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [ ]:
X_train.columns

Index(['vote_average', 'vote_count', 'runtime', 'budget', 'popularity', 'year',
       'month', 'budget_per_runtime', 'popularity_per_vote', 'budget_log',
       'vote_count_log', 'popularity_log', 'movie_age'],
      dtype='object')

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
model = Sequential()

In [ ]:
model.add(Dense(256, activation="relu", input_shape=(X_train.shape[1],)))
model.add(Dense(128, activation="relu"))
model.add(Dense(64, activation="relu"))
model.add(Dense(32, activation="relu"))
model.add(Dense(3, activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.7870 - loss: 0.4387 - val_accuracy: 0.8078 - val_loss: 0.3895
Epoch 2/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.8072 - loss: 0.3882 - val_accuracy: 0.8122 - val_loss: 0.3824
Epoch 3/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.8125 - loss: 0.3823 - val_accuracy: 0.8106 - val_loss: 0.3821
Epoch 4/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - accuracy: 0.8149 - loss: 0.3811 - val_accuracy: 0.8180 - val_loss: 0.3734
Epoch 5/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.8188 - loss: 0.3731 - val_accuracy: 0.8191 - val_loss: 0.3733
Epoch 6/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.8215 - loss: 0.3698 - val_accuracy: 0.8267 - val_loss: 0.3647
Epoch 7/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.8240 - loss: 0.3661 - val_accuracy: 0.8281 - val_loss: 0.3616
Epoch 8/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.8268 - loss: 0.3625 - val_accur

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

16685/16685 ━━━━━━━━━━━━━━━━━━━━ 33s 2ms/step - accuracy: 0.9473 - loss: 0.1344
Test Loss: 0.1344
Test Accuracy: 0.9473


In [ ]:
import numpy as np

y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

16685/16685 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.95      0.97    529016
           1       0.06      0.62      0.11      1949
           2       0.18      0.69      0.29      2954

    accuracy                           0.95    533919
   macro avg       0.41      0.75      0.46    533919
weighted avg       0.99      0.95      0.97    533919



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[502531  18105   8380]
 [    25   1218    706]
 [    32    889   2033]]


In [ ]:

model.save("/content/drive/MyDrive/movie_success_model.keras")


import joblib
joblib.dump(scaler, "/content/drive/MyDrive/movie_success_scaler.pkl")

['/content/drive/MyDrive/movie_success_scaler.pkl']